## Imports

In [0]:
from pyspark.sql import functions as F

import uuid
import time
import json
from faker import Faker

fake = Faker()

In [0]:
dbutils.widgets.dropdown("env", "dev", ["prod", "dev", "test"], "Environment")
ENV = dbutils.widgets.get("env")
dbutils.widgets.dropdown("catalog", "lakehouse_dev", ["lakehouse_prod", "lakehouse_dev"], "Data Catalog")
CATALOG = dbutils.widgets.get("catalog")


In [0]:
SCHEMA = "cloudwatch_metrics_pull"
BASE_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/raw_data/{ENV}"
USER_LANDING = f"{BASE_PATH}/users_json"
SPEND_LANDING = f"{BASE_PATH}/spend_csv"
csv_schema = "id,age,annual_income,spending_score"
print(BASE_PATH)

# List of directories to clean
directories = [
    f"{BASE_PATH}/users_json",
    f"{BASE_PATH}/spend_csv"
]


/Volumes/lakehouse_prod/cloudwatch_metrics_pull/raw_data/prod


## ⚠️ Cleanup LANDING ZONE ⚠️

In [0]:
print(directories)
def cleanup_landing_zone():
    
    for directory in directories:
        try:
            # recurse=True ensures all files and subdirectories are removed
            dbutils.fs.rm(directory, recurse=True)
            # Recreate the directory so the simulation script doesn't fail on missing paths
            dbutils.fs.mkdirs(directory)
            print(f"✅ Cleaned and recreated: {directory}")
        except Exception as e:
            print(f"⚠️ Error cleaning {directory}: {e}")

In [0]:
cleanup_landing_zone()

## ⚠️ DROP all assets ⚠️

In [0]:
def drop_medallion_tables():
    tables = [
        "user_bronze_sdp",
        "raw_user_data",
        "raw_spend_data",
        "user_silver_sdp",
        "spend_silver_sdp",
        "user_gold_sdp",
        "user_quarantine_sdp"
    ]

    print(f"Starting cleanup for schema: {CATALOG}.{SCHEMA}")

    for table in tables:
        table_path = f"{CATALOG}.{SCHEMA}.{table}"
        try:
            # spark.sql(f"DROP TABLE IF EXISTS {table_path}")
            print(f"✅ Successfully dropped table: {table_path}")
        except Exception as e:
            print(f"⚠️ Failed to drop {table_path}: {str(e)}")




In [0]:
# drop_medallion_tables()

## Kafka producer logic

In [0]:
def generate_coordinated_events(batch_size=5):
    user_events = []
    spend_events = []
    shared_ids = [fake.random_int(min=1, max=1000) for _ in range(batch_size)]
    spend_events.append(csv_schema)

    for user_id in shared_ids:
        user_events.append({
            "id": user_id,
            "email": fake.email(),
            # "creation_date": fake.date_time_this_year().strftime("%m-%d-%Y %H:%M:%S"),
            # "last_activity_date": fake.date_time_this_month().strftime("%m-%d-%Y %H:%M:%S"),
            "creation_date": fake.date_time_this_month().strftime("%m-%d-%Y %H:%M:%S"), # break expectation 'valid_activity_sequence'
            "last_activity_date": fake.date_time_this_year().strftime("%m-%d-%Y %H:%M:%S"), 
            "firstname": fake.first_name(),
            "lastname": fake.last_name(),
            "address": fake.address().replace("\n", ", "),
            "city": fake.city(),
            "last_ip": fake.ipv4(),
            "postcode": fake.postcode()
        })
        spend_events.append(
            f"{user_id},{fake.random_int(18, 90)},{round(fake.pyfloat(left_digits=3, right_digits=1, positive=True), 1)},{round(fake.pyfloat(left_digits=2, right_digits=1, positive=True), 1)}"
        )
    return user_events, spend_events


def produce_records(batch_size=10):
    user_data, spend_data = generate_coordinated_events(batch_size)
    unique_id = uuid.uuid4().hex[:8]
    timestamp = int(time.time())

    # User JSON
    user_json_str = "\n".join([json.dumps(e) for e in user_data])
    user_file = f"{USER_LANDING}/sim_users_{timestamp}_{unique_id}.json"
    dbutils.fs.put(user_file, user_json_str, overwrite=True)

    # Spend CSV (No header, matching the raw ingestion pattern)
    spend_csv_str = "\n".join(spend_data)
    spend_file = f"{SPEND_LANDING}/sim_spend_{timestamp}_{unique_id}.csv"
    dbutils.fs.put(spend_file, spend_csv_str, overwrite=True)

    print(f"Simulation Aligned with Test Data:")
    print(f" - Produced {batch_size} coordinated records.")
    print(f" - Users path: {user_file}")
    print(f" - Spend path: {spend_file}")




## Produce new data in Landing Zone

In [0]:
produce_records(2)

Wrote 612 bytes.
Wrote 70 bytes.
Simulation Aligned with Test Data:
 - Produced 2 coordinated records.
 - Users path: /Volumes/lakehouse_prod/cloudwatch_metrics_pull/raw_data/prod/users_json/sim_users_1775065820_fbbcc0eb.json
 - Spend path: /Volumes/lakehouse_prod/cloudwatch_metrics_pull/raw_data/prod/spend_csv/sim_spend_1775065820_fbbcc0eb.csv


## Check Landing Zone for new files

In [0]:
print("Checking Landing Zone Volumes...")
user_files = dbutils.fs.ls(f"{USER_LANDING}")
spend_files = dbutils.fs.ls(f"{SPEND_LANDING}")

print(f"Found {len(user_files)} user files and {len(spend_files)} spend files.")

# 2. Check Processing Status via Table Metadata
# This shows you the 'last_modified' time and row counts
tables = ["user_bronze_sdp", "spend_silver_sdp", "user_gold_sdp"]

for table in tables:
    full_name = f"{CATALOG}.{SCHEMA}.{table}"
    try:
        df = spark.table(full_name)
        count = df.count()
        # Get the latest record based on your simulated timestamps
        latest = df.select(F.max("_metadata.file_modification_time")).collect()[0][0] if "_metadata" in df.columns else "N/A"
        print(f"Table: {table} | Row Count: {count} | Latest File Processed: {latest}")
    except Exception as e:
        print(f"Could not read {full_name}: {e}")

# 3. Validation: Match User IDs between Bronze and Gold
# This confirms the join in 03-gold.py worked for your Faker data
bronze_ids = spark.table(f"{CATALOG}.{SCHEMA}.user_bronze_sdp").select("id").distinct()
gold_ids = spark.table(f"{CATALOG}.{SCHEMA}.user_gold_sdp").select("id").distinct()

match_count = gold_ids.intersect(bronze_ids).count()
print(f"Successful Joins in Gold Layer: {match_count}")

## Check Event Logs

In [0]:
%sql
DESCRIBE SELECT * FROM event_log("7eba62d9-8998-41c8-8cfb-d9ec9178e1a6")

### ALL

In [0]:
%sql
SELECT
  timestamp,
  -- Use dot notation to reach into the struct and array
  error.exceptions[0].message AS error_detail,

  -- Extract the specific rule name from the message string
  regexp_extract(error.exceptions[0].message, "Violated expectations: '([^']+)'", 1) AS violated_rule,

  -- Extract the input data snippet
  regexp_extract(error.exceptions[0].message, "Input data: '(.+)'", 1) AS offending_record,

  origin.pipeline_id
FROM event_log("7eba62d9-8998-41c8-8cfb-d9ec9178e1a6")
WHERE level = 'ERROR'
  -- Use the size function to ensure an exception actually exists before checking the message
  AND size(error.exceptions) > 0
  AND error.exceptions[0].message IS NOT NULL
  AND error.exceptions[0].message LIKE '%failed to meet the expectation%'
ORDER BY timestamp DESC

### WITHIN 20 MIN WINDOW

In [0]:
%sql
SELECT
  timestamp,
  -- Extracting the specific rule name
  regexp_extract(error.exceptions[0].message, "Violated expectations: '([^']+)'", 1) AS violated_rule,
  -- Extracting the record snippet for the CloudWatch log
  regexp_extract(error.exceptions[0].message, "Input data: '(.+)'", 1) AS offending_record,
  origin.pipeline_id,
  origin.update_id
FROM event_log("7eba62d9-8998-41c8-8cfb-d9ec9178e1a6")
WHERE level = 'ERROR'
  AND timestamp >= (current_timestamp() - INTERVAL 20 MINUTES)
  AND size(error.exceptions) > 0
  AND error.exceptions[0].message LIKE '%failed to meet the expectation%'
ORDER BY timestamp DESC

### Query cloudwatch_metrics_pull_events Table (configured in pipeline.yml)

In [0]:
spark.sql(
    f"""
      SELECT 
        timestamp,
        origin.update_id,
        event_type,
        message,
        details:flow_progress.metrics.num_output_rows as output_rows,
        details:flow_progress.data_quality.expectations as expectations
      FROM {CATALOG}.{SCHEMA}.cloudwatch_metrics_pull_events order by timestamp desc;
    """
).display()

## Check Quarantine Table for latest faulty records

In [0]:
spark.sql(f"""
SELECT 
  -- id, 
  -- quarantine_reason, 
quarantine_ts,
*
from {CATALOG}.{SCHEMA}.user_quarantine_sdp
where quarantine_ts >= (current_timestamp() - INTERVAL 15 MINUTES) 
order by quarantine_ts desc
""").display()

quarantine_ts,address,city,creation_date,email,firstname,id,last_activity_date,last_ip,lastname,postcode,_rescued_data,quarantine_reason,quarantine_ts
2026-04-01T17:50:37.786Z,"8877 Webster Bridge, Donaldfurt, AK 19112",Amandafort,04-01-2026 03:25:06,powellsherry@example.org,Evan,540,02-23-2026 17:00:37,50.43.164.81,Boone,43008,null,invalid_activity_sequence,2026-04-01T17:50:37.786Z
2026-04-01T17:50:37.786Z,"29008 Leblanc Union Apt. 632, Nicholeside, MN 04303",Ingramshire,04-01-2026 08:40:54,zrogers@example.net,Lisa,124,01-13-2026 05:11:54,62.170.102.163,Nelson,77364,null,invalid_activity_sequence,2026-04-01T17:50:37.786Z


## Adding the new rule to expectations

In [0]:
%sql
-- INSERT INTO lakehouse_dev.cloudwatch_metrics_pull.expectations (tag, name, constraint)
-- VALUES (
--   'user_silver_sdp',
--   'valid_activity_sequence',
--   'last_activity_date >= creation_date'
-- );

## Data Verification

In [0]:
spark.read.csv(f"/Volumes/{CATALOG}/cloudwatch_metrics_pull/raw_data/{ENV}/spend_csv/*.csv", header=True, inferSchema=True).display()

id,age,annual_income,spending_score
444,86,289.5,43.1
162,48,379.7,18.1
261,80,149.0,42.7
540,48,348.5,72.6
124,90,27.4,23.6


In [0]:
spark.read.json(f"/Volumes/{CATALOG}/cloudwatch_metrics_pull/raw_data/{ENV}/users_json/*.json").display()

address,city,creation_date,email,firstname,id,last_activity_date,last_ip,lastname,postcode
"06146 Peterson Loaf, Johnsonberg, LA 85081",South Haleystad,01-26-2026 20:28:12,uphillips@example.com,Jordan,444,04-01-2026 11:53:25,1.204.29.134,Baker,31904
"1548 Matthews Loop Apt. 968, Port Steven, MS 41616",Johnsonmouth,02-23-2026 07:28:41,matthew41@example.org,Robert,162,04-01-2026 11:04:22,76.197.44.191,Owen,03952
"42639 James Gateway, New Brett, WV 97103",East Barbarahaven,03-23-2026 09:42:08,agordon@example.com,Kayla,261,04-01-2026 09:54:12,213.188.204.38,Moore,22973
"8877 Webster Bridge, Donaldfurt, AK 19112",Amandafort,04-01-2026 03:25:06,powellsherry@example.org,Evan,540,02-23-2026 17:00:37,50.43.164.81,Boone,43008
"29008 Leblanc Union Apt. 632, Nicholeside, MN 04303",Ingramshire,04-01-2026 08:40:54,zrogers@example.net,Lisa,124,01-13-2026 05:11:54,62.170.102.163,Nelson,77364


In [0]:
df = spark.table(f"{CATALOG}.{SCHEMA}.raw_user_data")
display(df)

address,city,creation_date,email,firstname,id,last_activity_date,last_ip,lastname,postcode,_rescued_data
"06146 Peterson Loaf, Johnsonberg, LA 85081",South Haleystad,01-26-2026 20:28:12,uphillips@example.com,Jordan,444,04-01-2026 11:53:25,1.204.29.134,Baker,31904,null
"1548 Matthews Loop Apt. 968, Port Steven, MS 41616",Johnsonmouth,02-23-2026 07:28:41,matthew41@example.org,Robert,162,04-01-2026 11:04:22,76.197.44.191,Owen,03952,null
"42639 James Gateway, New Brett, WV 97103",East Barbarahaven,03-23-2026 09:42:08,agordon@example.com,Kayla,261,04-01-2026 09:54:12,213.188.204.38,Moore,22973,null
"8877 Webster Bridge, Donaldfurt, AK 19112",Amandafort,04-01-2026 03:25:06,powellsherry@example.org,Evan,540,02-23-2026 17:00:37,50.43.164.81,Boone,43008,null
"29008 Leblanc Union Apt. 632, Nicholeside, MN 04303",Ingramshire,04-01-2026 08:40:54,zrogers@example.net,Lisa,124,01-13-2026 05:11:54,62.170.102.163,Nelson,77364,null


In [0]:
df = spark.table(f"{CATALOG}.{SCHEMA}.raw_spend_data")
display(df)

id,age,annual_income,spending_score,_rescued_data
444,86,289.5,43.1,null
162,48,379.7,18.1,null
261,80,149.0,42.7,null
540,48,348.5,72.6,null
124,90,27.4,23.6,null


In [0]:
df = spark.table(f"{CATALOG}.{SCHEMA}.user_bronze_sdp")
display(df)

address,city,creation_date,email,firstname,id,last_activity_date,last_ip,lastname,postcode,_rescued_data
"06146 Peterson Loaf, Johnsonberg, LA 85081",South Haleystad,01-26-2026 20:28:12,uphillips@example.com,Jordan,444,04-01-2026 11:53:25,1.204.29.134,Baker,31904,null
"1548 Matthews Loop Apt. 968, Port Steven, MS 41616",Johnsonmouth,02-23-2026 07:28:41,matthew41@example.org,Robert,162,04-01-2026 11:04:22,76.197.44.191,Owen,03952,null
"42639 James Gateway, New Brett, WV 97103",East Barbarahaven,03-23-2026 09:42:08,agordon@example.com,Kayla,261,04-01-2026 09:54:12,213.188.204.38,Moore,22973,null
"8877 Webster Bridge, Donaldfurt, AK 19112",Amandafort,04-01-2026 03:25:06,powellsherry@example.org,Evan,540,02-23-2026 17:00:37,50.43.164.81,Boone,43008,null
"29008 Leblanc Union Apt. 632, Nicholeside, MN 04303",Ingramshire,04-01-2026 08:40:54,zrogers@example.net,Lisa,124,01-13-2026 05:11:54,62.170.102.163,Nelson,77364,null


In [0]:
df = spark.table(f"{CATALOG}.{SCHEMA}.user_silver_sdp")
display(df)

id,email,creation_date,last_activity_date,firstname,lastname,address,city,last_ip,postcode
444,ea74edcbaa6a8d0ec94b5f91c9652b90ffd1be6b,2026-01-26T20:28:12.000Z,2026-04-01T11:53:25.000Z,Jordan,Baker,"06146 Peterson Loaf, Johnsonberg, LA 85081",South Haleystad,1.204.29.134,31904
162,ab98c7842371b45b4ce11652051754a775b345c7,2026-02-23T07:28:41.000Z,2026-04-01T11:04:22.000Z,Robert,Owen,"1548 Matthews Loop Apt. 968, Port Steven, MS 41616",Johnsonmouth,76.197.44.191,03952
261,7f5e722eff4cd23b851cdd658f2e68605b3e1c0a,2026-03-23T09:42:08.000Z,2026-04-01T09:54:12.000Z,Kayla,Moore,"42639 James Gateway, New Brett, WV 97103",East Barbarahaven,213.188.204.38,22973


In [0]:
df = spark.table(f"{CATALOG}.{SCHEMA}.spend_silver_sdp")
display(df)

id,age,annual_income,spending_score,_rescued_data,spending_efficiency
444,86,289.5,43.1,null,14.887736951337166
162,48,379.7,18.1,null,4.766921200834914
261,80,149.0,42.7,null,28.65771863284527
540,48,348.5,72.6,null,20.83213729530017
124,90,27.4,23.6,null,86.13138945268213


In [0]:
spark.table(f"{CATALOG}.{SCHEMA}.user_quarantine_sdp").display()


address,city,creation_date,email,firstname,id,last_activity_date,last_ip,lastname,postcode,_rescued_data,quarantine_reason,quarantine_ts
"8877 Webster Bridge, Donaldfurt, AK 19112",Amandafort,04-01-2026 03:25:06,powellsherry@example.org,Evan,540,02-23-2026 17:00:37,50.43.164.81,Boone,43008,null,invalid_activity_sequence,2026-04-01T17:50:37.786Z
"29008 Leblanc Union Apt. 632, Nicholeside, MN 04303",Ingramshire,04-01-2026 08:40:54,zrogers@example.net,Lisa,124,01-13-2026 05:11:54,62.170.102.163,Nelson,77364,null,invalid_activity_sequence,2026-04-01T17:50:37.786Z


In [0]:
spark.table(f"{CATALOG}.{SCHEMA}.user_gold_sdp").display()

id,email,creation_date,last_activity_date,firstname,lastname,address,city,last_ip,postcode,age,annual_income,spending_score,_rescued_data,spending_efficiency
444,ea74edcbaa6a8d0ec94b5f91c9652b90ffd1be6b,2026-01-26T20:28:12.000Z,2026-04-01T11:53:25.000Z,Jordan,Baker,"06146 Peterson Loaf, Johnsonberg, LA 85081",South Haleystad,1.204.29.134,31904,86,289.5,43.1,null,14.887736951337166
162,ab98c7842371b45b4ce11652051754a775b345c7,2026-02-23T07:28:41.000Z,2026-04-01T11:04:22.000Z,Robert,Owen,"1548 Matthews Loop Apt. 968, Port Steven, MS 41616",Johnsonmouth,76.197.44.191,03952,48,379.7,18.1,null,4.766921200834914
261,7f5e722eff4cd23b851cdd658f2e68605b3e1c0a,2026-03-23T09:42:08.000Z,2026-04-01T09:54:12.000Z,Kayla,Moore,"42639 James Gateway, New Brett, WV 97103",East Barbarahaven,213.188.204.38,22973,80,149.0,42.7,null,28.65771863284527


### Verify Gold Table logic

In [0]:
spark.table(f"{CATALOG}.{SCHEMA}.user_silver_sdp").join(spark.read.table(f"{CATALOG}.{SCHEMA}.spend_silver_sdp"), ["id"], "left").display()


id,email,creation_date,last_activity_date,firstname,lastname,address,city,last_ip,postcode,age,annual_income,spending_score,_rescued_data,spending_efficiency
444,ea74edcbaa6a8d0ec94b5f91c9652b90ffd1be6b,2026-01-26T20:28:12.000Z,2026-04-01T11:53:25.000Z,Jordan,Baker,"06146 Peterson Loaf, Johnsonberg, LA 85081",South Haleystad,1.204.29.134,31904,86,289.5,43.1,null,14.887736951337166
162,ab98c7842371b45b4ce11652051754a775b345c7,2026-02-23T07:28:41.000Z,2026-04-01T11:04:22.000Z,Robert,Owen,"1548 Matthews Loop Apt. 968, Port Steven, MS 41616",Johnsonmouth,76.197.44.191,03952,48,379.7,18.1,null,4.766921200834914
261,7f5e722eff4cd23b851cdd658f2e68605b3e1c0a,2026-03-23T09:42:08.000Z,2026-04-01T09:54:12.000Z,Kayla,Moore,"42639 James Gateway, New Brett, WV 97103",East Barbarahaven,213.188.204.38,22973,80,149.0,42.7,null,28.65771863284527


## Expectations

In [0]:

data = [
 # tag/table name      name              constraint
 ("user_bronze_sdp",  "correct_schema", "_rescued_data IS NULL"),
 ("user_silver_sdp",  "valid_id",       "id IS NOT NULL AND id > 0"),
 ("user_silver_sdp",  "valid_activity_sequence", "last_activity_date >= creation_date"),
 ("spend_silver_sdp", "valid_id",       "id IS NOT NULL AND id > 0"),
 ("user_gold_sdp",    "valid_age",      "age IS NOT NULL"),
 ("user_gold_sdp",    "valid_income",   "annual_income IS NOT NULL"),
 ("user_gold_sdp",    "valid_score",    "spending_score IS NOT NULL")
]
# spark.createDataFrame(data=data, schema=["tag", "name", "constraint"]).write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.expectations")

In [0]:
%sql
-- drop table lakehouse_dev.cloudwatch_metrics_pull.expectations